# Chapter 15: Build Your Own Lift Engine

This notebook runs every piece of code in the chapter, in order, with the chapter's text around it. Read the chapter on the site: [Build Your Own Lift Engine](https://books.sugiv.fyi/lift/lift-engine.html). Everything is simulated from seeded code in `sim/`, so a run gives the same numbers as the book.

In [1]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the
# notebook from a clone of it. Either way it moves to the repo's root so `sim` can be imported.
import os
import pathlib
import sys

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/lift-not-luck").exists():
        !git clone -q --depth 1 https://github.com/sugix/lift-not-luck /content/lift-not-luck
    %pip install -q pingouin
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/lift-not-luck")]
root = next(p for p in candidates if (p / "sim" / "parlo.py").is_file())
os.chdir(root)
sys.path.insert(0, str(root))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
def glue(name, value, display=True):
    """In the book, glue() puts a computed number into the text. Here the numbers are already
    written into the text, so it only shows the value when asked to."""
    if display:
        from IPython.display import display as show
        show(value)

from sim import engine as G, voicemode as V, style

style.apply()
pd.set_option("display.precision", 4)


def glue_num(name, value):
    glue(name, float(value), display=False)

Here's the finding first. Everything this book has taught about reading an experiment fits in about **two hundred lines of Python**: `lift/sim/engine.py`. Given one table with a row per learner, it checks the split, estimates every metric on the right unit, tests the guardrails, corrects for many metrics and writes a decision memo. Run on Juno's **Voice Mode** test, the memo says the feature works, and also that it shouldn't ship yet: it costs too much per lesson. And because the engine is code, it can be tested like code: across 150 A/A tests and twelve metrics, its false-alarm rate is in line with the promised 5%, and its confidence intervals cover the true lift about 95% of the time.

You may never need to build your own. GrowthBook, Statsig, Eppo and the in-house platforms at large companies do all of this and more. But building a small one, once, is the best way I know to understand what those platforms are doing, to sanity-check them, and to analyse the experiments they don't cover: agent pipelines, offline evaluations, one-off tests run from a notebook.

> **Note**
>
> The engine is `lift/sim/engine.py` and the demo experiment is `lift/sim/voicemode.py`. Both are short enough to read in one sitting, which is the point.

## The shape of an engine

Every experimentation platform has the same pipeline, whatever it's called:

| Step | Question | Chapter |
|---|---|---|
| **Assign** | Who's in which arm, consistently? | 1, 2 |
| **Log exposure** | Who actually saw their arm, and when? | 3 |
| **Aggregate per unit** | One row per learner (the unit you randomised) | 2 |
| **Check** | Is the split what we asked for, everywhere? | 3 |
| **Estimate** | Lift and uncertainty, with CUPED and ratio metrics | 2, 5 |
| **Guard** | Did anything we promised not to break, break? | 2 |
| **Correct** | Many metrics, one false-alarm budget | 6 |
| **Decide** | Ship, don't ship, or keep going | 4 |

The engine covers every step except logging and per-learner aggregation, which live in your app and warehouse.

## Step 1: assignment you can recompute

Randomising with `random()` at request time works once. It breaks the second time the same learner shows up, because they might get the other arm. Real systems **hash**: combine the experiment's name (the *salt*) with the learner's id, hash it, and turn the hash into a number between 0 and 1. The same learner always gets the same number for the same experiment, on any server, with no database lookup.

In [3]:
ids = [f"u{i:06d}" for i in range(30_000)]
arm = G.assign(ids, salt="juno-voice-mode")
other = G.assign(ids, salt="streak-shield-v3")
print(f"share in treatment: {(arm == 'treatment').mean():.2%}")
print(f"same learner, asked again: {G.assign(ids[:3], 'juno-voice-mode')} == {arm[:3]}")
pd.crosstab(pd.Series(arm, name="Voice Mode"), pd.Series(other, name="Streak Shield v3"))

share in treatment: 49.99%
same learner, asked again: ['control' 'control' 'treatment'] == ['control' 'control' 'treatment']


Streak Shield v3,control,treatment
Voice Mode,,
control,7413,7591
treatment,7325,7671


Three properties, all visible above: the split is balanced, it's **consistent** (asking again gives the same answer), and a different salt gives an **independent** split, so two experiments running at once don't line up their arms. That last one matters: if both experiments used the same salt, every Voice Mode learner would also be in the same Streak Shield arm, and their effects would be inseparable.

## Step 2: metrics as configuration

Metrics are declared once, with the details that decide how they're analysed: what's summed per learner, whether it's a ratio, which pre-period column CUPED can use, the metric's role and, for guardrails, the margin.

In [4]:
metrics = [
    G.Metric("lessons per learner", "lessons", covariate="pre_lessons", role="decision"),
    G.Metric("latency per turn", "latency_total_s", "turns", role="guardrail",
             higher_is_better=False, margin=0.10),
    G.Metric("cost per lesson", "cost_usd", "lessons", role="guardrail",
             higher_is_better=False, margin=0.10),
    G.Metric("completion per lesson", "lessons_completed", "lessons"),
    G.Metric("minutes studied", "minutes"),
] + [G.Metric(name.replace("_", " "), name) for name in V.SECONDARY_NULL]

voice = G.Experiment("Juno Voice Mode", metrics, segments=["platform"])

This list *is* Chapter 2's one-page design, in code. Writing it before the test starts is what stops anyone choosing the decision metric after seeing the results. The Juno metrics, latency and cost, sit next to the product metrics. For an agent feature they're not an afterthought; they're half the decision.

## Step 3: check before you read

The engine refuses to read a result until the sample ratio checks pass, overall and in every segment. Here's Voice Mode with Chapter 3's bug planted: the new voice screen crashes on some Android phones before exposure is logged.

In [5]:
broken = V.simulate(arm=arm, crash_share=0.08)
print(G.decide(broken, voice))
G.srm_table(broken, voice).round(4)

STOP: sample ratio mismatch in platform = Android. Find the cause before reading any metric.


,users,treatment share,p_value,fails (p < 0.001)
slice,,,,
overall,29540,0.4921,0.0065,False
platform = Android,11493,0.4818,0.0001,True
platform = iOS,13538,0.4958,0.3272,False
platform = web,4509,0.5072,0.3330,False


The memo stops at the first line. Overall, the split is only slightly off; inside Android it's clearly broken. That's the whole point of checking by segment. Fix the crash, rerun:

## Step 4: the readout

In [6]:
data = V.simulate(arm=arm)
report = G.readout(data, voice)
cols = ["role", "control", "treatment", "lift", "ci_low", "ci_high", "p_value", "p_bh", "guardrail"]
report[cols].style.format({"control": "{:.3f}", "treatment": "{:.3f}", "lift": "{:+.1%}",
                           "ci_low": "{:+.1%}", "ci_high": "{:+.1%}", "p_value": "{:.4f}",
                           "p_bh": "{:.3f}"}, na_rep="")

,role,control,treatment,lift,ci_low,ci_high,p_value,p_bh,guardrail
metric,,,,,,,,,
lessons per learner,decision,8.711,8.943,+2.7%,+1.2%,+4.1%,0.0003,,
latency per turn,guardrail,1.472,1.551,+5.4%,+4.2%,+6.6%,0.0000,,pass
cost per lesson,guardrail,0.006,0.007,+14.4%,+13.2%,+15.6%,0.0000,,fail
completion per lesson,secondary,0.737,0.750,+1.8%,+1.1%,+2.5%,0.0000,0.000,
minutes studied,secondary,19.188,20.358,+6.1%,+3.6%,+8.5%,0.0000,0.000,
streak days,secondary,1.530,1.552,+1.4%,-1.1%,+4.0%,0.2751,0.563,
vocab reviews,secondary,2.302,2.329,+1.2%,-1.2%,+3.5%,0.3314,0.563,
shares,secondary,3.050,3.081,+1.0%,-1.2%,+3.2%,0.3753,0.563,
ratings given,secondary,3.834,3.841,+0.2%,-1.9%,+2.3%,0.8613,0.960,


Each row was computed the way an earlier chapter said it should be:

- **Lessons per learner** uses CUPED on pre-period lessons (Chapter 5).
- **Latency per turn, cost per lesson and completion per lesson** are ratios analysed per learner with the delta method (Chapter 2), not per turn or per lesson.
- **Guardrails** are non-inferiority tests against a 10% margin (Chapter 2): "pass" means the data rule out a harm that big.
- **Secondary metrics** get Benjamini-Hochberg adjusted p-values (`p_bh`, Chapter 6). Only two survive. The other seven moved by chance or not at all.

How close are these to the truth? The simulator can compute each metric's real lift by running the whole population through both arms:

In [7]:
everyone_t = V.simulate(n=300_000, arm=np.full(300_000, "treatment"), seed=99)
everyone_c = V.simulate(n=300_000, arm=np.full(300_000, "control"), seed=99)


def true_lift(m):
    if m.denominator is None:
        return everyone_t[m.numerator].mean() / everyone_c[m.numerator].mean() - 1
    rt = everyone_t[m.numerator].sum() / everyone_t[m.denominator].sum()
    rc = everyone_c[m.numerator].sum() / everyone_c[m.denominator].sum()
    return rt / rc - 1


truth = pd.Series({m.name: true_lift(m) for m in metrics})
check = report[["lift", "ci_low", "ci_high"]].assign(true_lift=truth)
check["CI covers truth"] = (check.ci_low <= check.true_lift) & (check.true_lift <= check.ci_high)
check.head(5).style.format({"lift": "{:+.1%}", "ci_low": "{:+.1%}", "ci_high": "{:+.1%}", "true_lift": "{:+.1%}"})

,lift,ci_low,ci_high,true_lift,CI covers truth
metric,,,,,
lessons per learner,+2.7%,+1.2%,+4.1%,+3.6%,True
latency per turn,+5.4%,+4.2%,+6.6%,+6.0%,True
cost per lesson,+14.4%,+13.2%,+15.6%,+15.1%,True
completion per lesson,+1.8%,+1.1%,+2.5%,+1.9%,True
minutes studied,+6.1%,+3.6%,+8.5%,+5.2%,True


## Step 5: the decision memo

In [8]:
print(G.decide(data, voice))

Experiment: Juno Voice Mode
Decision metric 'lessons per learner': +2.7% (95% CI +1.2% to +4.1%), chance to win 100%, expected loss if shipped 0.00%.
Guardrail 'latency per turn': +5.4% (95% CI +4.2% to +6.6%), pass.
Guardrail 'cost per lesson': +14.4% (95% CI +13.2% to +15.6%), fail.
Secondary metrics moving after FDR correction: completion per lesson, minutes studied.
RECOMMENDATION: do not ship as is. Guardrail(s) not shown to be safe: cost per lesson.


In [9]:
d = report.loc["lessons per learner"]
c = report.loc["cost per lesson"]
glue_num("dec_lift", 100 * d.lift)
glue_num("cost_lift", 100 * c.lift)
glue_num("cost_lo", 100 * c.ci_low)

Voice Mode works: lessons per learner are up 2.7%, with essentially no chance it's a loss. But cost per lesson is up 14%, and even the bottom of its confidence interval (13%) is above the 10% the team promised. So the memo says **don't ship as is**.

That's the most useful kind of result, because it says what to fix. Voice turns use more tokens (transcription, then the tutor, then speech). The team can cache common phrases, use a smaller speech model, or reconsider the margin with the finance team, *explicitly*, and then test again. What it shouldn't do is quietly drop the guardrail because the headline metric looks good.

The memo's logic is short enough to read in the source: stop on any sample ratio failure; report the decision metric with its chance to win and expected loss (Chapter 4); block on any failed guardrail; otherwise ship when the expected loss is below the threshold of caring.

## Step 6: test the engine like code

An analysis engine is code, and code has bugs. The best test is the one from Chapter 3: run it on many **A/A tests**, where nothing changes, and check that every metric's false-alarm rate is 5%. Then run it on tests with known effects and check that the confidence intervals cover the truth 95% of the time.

In [10]:
reps = 150
alarms = pd.DataFrame(0.0, index=[m.name for m in metrics], columns=["A/A false alarms"])
for s in range(reps):
    aa = V.simulate(n=10_000, effects={}, seed=1_000 + s)
    alarms["A/A false alarms"] += (G.readout(aa, voice).p_value < 0.05).astype(float)
alarms /= reps

covered = pd.Series(0.0, index=[m.name for m in metrics])
for s in range(reps):
    ab = V.simulate(n=10_000, seed=5_000 + s)
    r = G.readout(ab, voice)
    covered += ((r.ci_low <= truth) & (truth <= r.ci_high)).astype(float)
alarms["CI covers the true lift"] = covered / reps
alarms.style.format("{:.0%}")

,A/A false alarms,CI covers the true lift
lessons per learner,1%,98%
latency per turn,2%,91%
cost per lesson,2%,95%
completion per lesson,3%,94%
minutes studied,5%,97%
streak days,6%,98%
vocab reviews,2%,97%
shares,3%,98%
ratings given,6%,98%
settings changes,9%,99%


In [11]:
glue_num("fa_min", 100 * alarms["A/A false alarms"].min())
glue_num("fa_max", 100 * alarms["A/A false alarms"].max())
glue_num("cov_min", 100 * alarms["CI covers the true lift"].min())
glue_num("cov_max", 100 * alarms["CI covers the true lift"].max())
glue_num("fa_pooled", 100 * alarms["A/A false alarms"].mean())
glue_num("cov_pooled", 100 * alarms["CI covers the true lift"].mean())

Pooled across all twelve metrics, the false-alarm rate is 4.0% and the intervals cover the truth 97% of the time. Individual metrics range from 1% to 9% false alarms and 91% to 99% coverage, which is about what 150 tests give by chance alone: with true rates of 5% and 95%, anything from about 1% to 9%, and about 91% to 99%, is normal. To pin each metric down tightly, run a few thousand A/A tests overnight. If someone "optimises" the engine by analysing ratio metrics per lesson instead of per learner, the A/A false-alarm rate for those metrics jumps, as Chapter 2 showed, and this test catches it. **Put it in CI.** It runs in seconds.

## What a platform adds

The engine is the statistical core. A production platform wraps it in the things that make it safe for a whole company to use:

- **A metric layer.** Metrics defined once in SQL against the warehouse, versioned and reviewed, so every team computes "lessons per learner" the same way.
- **Exposure logging and assignment services.** The hash in step 1, served at low latency, with exposure events logged when the learner actually sees their arm.
- **Scheduled readouts and always-valid monitoring.** Daily updates with Chapter 5's sequential tests, so peeking is safe and guardrails can stop a test early.
- **Holdouts and bandits.** Long-term holdouts (Chapter 10) and bandit allocation (Part II) built in, with propensities logged.
- **Heterogeneity.** Segment breakdowns with correction, and uplift models (Chapter 14) for targeting.

The tools that do this come in two broad kinds, and it helps to know which one you're looking at. **Experimentation platforms**, such as GrowthBook, Statsig, Eppo and Optimizely, and the in-house systems at large companies, run tests and read them out for whatever you change. **Decisioning platforms** go further and make the choices themselves. JustAI, the company behind the Better Mortgage case study in Part II, is one example for lifecycle messaging: by its own public description, the system picks each customer's next message and its order from a shared record of customer context, tests combinations of messages, and reports lift against a holdout. That's this book's bandits and holdouts in product form: an algorithm choosing, and a holdout proving. I mention it to show where the book's methods sit inside real products, not to recommend any vendor. Whatever the tool, ask it the questions this book taught: what's the unit, where's the holdout, are propensities logged, and how are guardrails tested?

For agent teams, two additions matter most. **Log decision inputs and probabilities** for every model call, tool call and route (Chapters 9, 10 and 11): it turns every production decision into usable evidence. And **treat offline evaluations as experiments**: an eval run over a fixed set of prompts is a paired test, and it deserves the same checks, units and corrections as anything else in this book.

> **In the wild**
>
> Kohavi, Tang and Xu (Kohavi et al., 2020), who between them built or ran experimentation at Microsoft, Google and LinkedIn, describe how such platforms are put together. They share this chapter's skeleton: stable hashing for assignment, a central metric definition layer, automatic sample ratio checks, and readouts that refuse to show results when a check fails.

## Pitfalls

- **Random assignment at request time.** Learners flip between arms. Hash a stable id with a per-experiment salt.
- **One salt for every experiment.** Arms line up across experiments, and effects become inseparable.
- **Reading metrics before the checks.** Make the engine refuse, as `decide` does.
- **Untested analysis code.** A/A tests are unit tests for statistics. Run them on every change.
- **Building the platform before the habits.** The hard parts are cultural: deciding metrics in advance, respecting guardrails, reporting flat results. An engine can enforce some of that; it can't create it.

## Exercises

1. **A sequential guardrail.** Add Chapter 5's always-valid confidence sequence to `readout`, and make `decide` stop early when a guardrail is shown to be harmful.
2. **Uneven splits.** Run Voice Mode as a 10/90 test: pass `treatment_share=0.1` to both `G.assign` and `G.Experiment`. What changes in the readout and the sample ratio check?
3. **Break it on purpose.** Change `_arm_stats` to compute ratio metrics per lesson instead of per learner. Rerun the A/A test. Which metrics' false-alarm rates move?
4. **An agent eval.** Juno's team runs 400 fixed prompts through two prompt versions and scores each answer. Adapt the engine to a paired design, where each prompt is its own control.

## Further reading

Kohavi, Tang and Xu (Kohavi et al., 2020) describe how experimentation platforms at Microsoft, Google and LinkedIn are built and run, including the cultural side. Deng, Knoblich and Lu (Deng et al., 2018) cover the delta method as platforms use it. Fabijan and colleagues (Fabijan et al., 2019) describe automated sample ratio checks in production.